**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**05 | Frozen Test and Research Synthesis**

![Frozen Test and Research Synthesis](../../images/05_frozen_test_and_research_synthesis.svg)

**Objective:** Apply the Development-selected IMU event detector to unseen participants once, quantify event timing and coverage, and interpret participant-level uncertainty.

**Dataset:** Camargo et al. (2021), source-validated original-time paired IMU and independent force-derived contact references.

**Notebook scope:** Development/Test boundaries and stated measurement units are preserved. Numerical results are only established after original-data execution.


**Current exploratory pathway:** Frozen participant-disjoint Test evaluation against original Camargo `gcRight` phase-annotation event times. This is *annotation agreement*, not verified ground-truth force-based initial-contact accuracy or clinical external validation. The selected Development model, exact configuration and input provenance are checked before evaluation.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
# This IPython-native checkout requires no imports outside Section 01.
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = "/content/human-movement-intelligence"
shell = get_ipython()
if shell is None:
    raise RuntimeError("Execute this notebook in Google Colab or IPython.")
shell.system(
    f'if [ ! -d "{repo_path}" ]; then git clone -q "{repo_url}" '
    f'"{repo_path}"; fi'
)

# ---------------------------- 2. Open Repository -----------------------------
shell.run_line_magic("cd", repo_path)


**01. Imports and Research Environment**

Load the saved Development-selected configuration and the participant-disjoint Test records. No tuning or source-dependent fitting occurs in this Notebook.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 5

# ----------------- 2. Scientific Computing and Data Analysis -----------------
from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 4. Study Research Utilities ------------------------
from gait_experiments import (load_trial_collection,
    evaluate_trial_collection, FIXED_BASELINE)
from gait_evaluation import (participant_metrics, paired_subject_bootstrap,
    source_quality_sensitivity)

# ---------------------- 5. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
output_dir = study_dir / "output"
ANALYSIS_MODE = "source_phase_annotation"  # Explicit exploratory pilot
if ANALYSIS_MODE == "source_phase_annotation":
    cache_dir = root / "data/processed/camargo/phase_annotation_trials"
    prefix = "05_phase"
    event_key = "annotation_ic_s"
    TARGET = "original_gcRight_HeelStrike_phase_wrap"
    selected_path = output_dir / "04_phase_selected_configuration.csv"
    source_trials_path = output_dir / "02_phase_annotation_trials.csv"
    roles_path = output_dir / "02_phase_participant_roles.csv"
elif ANALYSIS_MODE == "validated_force":
    cache_dir = root / "data/processed/camargo/paired_trials"
    prefix = "05"
    event_key = "reference_ic_s"
    TARGET = "verified_independent_force_IC"
    selected_path = output_dir / "04_selected_configuration.csv"
    source_trials_path = output_dir / "02_eligible_trials.csv"
    roles_path = output_dir / "02_participant_roles.csv"
else:
    raise ValueError("Unrecognised analysis mode.")
selection = pd.read_csv(selected_path)
if len(selection) != 1:
    raise ValueError("Notebook 04 must save exactly one selected configuration.")
choice = selection.iloc[0]
model_name = str(choice.selected_model_id)
config = json.loads(choice.configuration_json)
if str(choice.target_provenance) != TARGET:
    raise ValueError("Frozen Development configuration targets a different label.")
independent_force_reference_verified = (
    ANALYSIS_MODE == "validated_force"
)
if bool(choice.independent_force_reference_verified) != (
        independent_force_reference_verified):
    raise ValueError("Frozen model event provenance disagrees with current mode.")

tolerance = float(choice.match_tolerance_s)
print("Development-selected recipe:", model_name, config)
print("Matching tolerance:", tolerance, "seconds")


**02. Verify Frozen Participant Roles and Source Inputs**

Test includes only participants reserved in Notebook 02, and their original-time numerical inputs must already exist. Neither model choice nor detection threshold is adjusted after inspecting Test scores.


In [ ]:
# ========================= 02. Verify Reserved Test ==========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
eligible = pd.read_csv(
    source_trials_path, dtype={"subject_id": str, "trial_key": str}
)
roles = pd.read_csv(roles_path, dtype={"subject_id": str})
if (eligible.trial_key.duplicated().any()
        or roles.subject_id.duplicated().any()):
    raise ValueError("Duplicate eligible trials or participant roles.")
test_people = set(roles.loc[roles.role=="Test", "subject_id"])
development_people = set(roles.loc[roles.role=="Development","subject_id"])
if not test_people or (test_people & development_people):
    raise ValueError("Missing or overlapping participant roles.")
test = eligible.loc[eligible.subject_id.isin(test_people)].copy()
if set(test.subject_id) != test_people:
    raise ValueError("Some reserved Test people have no eligible source trials.")
trials = load_trial_collection(cache_dir, test, event_key=event_key)
print("Held-out participants:", sorted(test_people),
      "eligible Test trials:", len(trials))
print("Evaluation target:", TARGET)
if ANALYSIS_MODE == "source_phase_annotation":
    print("Participant-disjoint source-annotation pilot; "
          "NOT independent force validation or clinical generalisation.")


**03. Evaluate Frozen Model and Fixed Reference Baseline**

Both detectors are scored on the identical reserved Test trials. The primary measurement is the selected model's participant-mean matched IC timing MAE with explicit sensitivity, precision, misses and extras. B0 remains a predefined descriptive comparator.


In [ ]:
# ======================== 03. Evaluate Frozen Models =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
results = []
# Compare two distinct recipes only when Development selected B1.
methods = [("B0_fixed", FIXED_BASELINE)]
if model_name != "B0_fixed":
    methods.append(("selected_" + model_name, config))
for model_id, method in methods:
    trial_scores = evaluate_trial_collection(
        trials, method, model_id, tolerance
    )
    results.append(trial_scores)
test_scores = pd.concat(results, ignore_index=True)
if ANALYSIS_MODE == "source_phase_annotation":
    if "review_source_quality" not in test.columns:
        raise ValueError("Source trial quality flag is missing.")
    test_scores = test_scores.merge(
        test[["trial_key", "review_source_quality"]].drop_duplicates(),
        on="trial_key", how="left", validate="many_to_one",
    )
    if test_scores.review_source_quality.isna().any():
        raise ValueError("Source trial quality flags are incomplete.")
    test_scores["review_source_quality"] = (
        test_scores.review_source_quality.astype(bool)
    )
test_scores["target_provenance"] = TARGET
test_scores["independent_force_reference_verified"] = (
    independent_force_reference_verified
)

# ------------------------- 2. Save Scientific Tables -------------------------
test_scores.to_csv(output_dir / f"{prefix}_test_trial_metrics.csv", index=False)
display(test_scores.head())


**04. Calculate Person-Balanced Test Evidence**

Summarise at the independent participant level, not the number of events. Timing MAE must always be interpreted with the event-detection counts; no matched events produce undefined timing error rather than a spurious zero.


In [ ]:
# ====================== 04. Summarise Test Participants ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
p = []
for model_id, frame in test_scores.groupby("model_id"):
    p.append(participant_metrics(frame).assign(model_id=model_id))
person_results = pd.concat(p, ignore_index=True)
person_results["target_provenance"] = TARGET
person_results["independent_force_reference_verified"] = (
    independent_force_reference_verified
)
person_results.to_csv(
    output_dir / f"{prefix}_test_participant_metrics.csv", index=False
)
summary = (
    person_results.groupby("model_id", as_index=False)
        .agg(mean_subject_timing_mae_ms=("timing_mae_ms", "mean"),
             mean_subject_sensitivity=("sensitivity", "mean"),
             mean_subject_precision=("precision", "mean"),
             n_participants=("subject_id", "nunique"))
)

# ------------------------- 2. Save Scientific Tables -------------------------
summary.to_csv(output_dir / f"{prefix}_test_model_summary.csv", index=False)
display(summary)
if ANALYSIS_MODE == "source_phase_annotation":
    source_qc = source_quality_sensitivity(test_scores)
    source_qc.to_csv(
        output_dir / "05_phase_source_quality_sensitivity.csv",
        index=False,
    )
    print("Post-hoc source quality sensitivity (not used to select a model):")
    display(source_qc)



**05. Paired Uncertainty and Event Error Diagnostics**

Describe paired whole-person uncertainty. A small number of Test participants limits precision even when many correlated strides or events are recorded. No inferential superiority is asserted from descriptive intervals.


In [ ]:
# ======================== 05. Paired Test Uncertainty ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
if model_name == "B0_fixed":
    # No paired model effect exists when the fixed baseline was selected.
    uncertainty = pd.DataFrame([{
        "comparator": "B0_fixed", "selected_model": model_name,
        "n_participants": len(test_people), "mean_difference_ms": np.nan,
        "lower_ms": np.nan, "upper_ms": np.nan,
        "comparison_note": "Same Development-selected model; no paired contrast"
    }])
else:
    delta = paired_subject_bootstrap(
        person_results, "B0_fixed", "selected_" + model_name,
        draws=2000, seed=42
    )
    uncertainty = pd.DataFrame([{
        "comparator": "B0_fixed", "selected_model": model_name,
        "comparison_note": "Selected B1 minus fixed B0 timing MAE", **delta
    }])

# ------------------------- 2. Save Scientific Tables -------------------------
uncertainty.to_csv(
    output_dir / f"{prefix}_paired_participant_uncertainty.csv", index=False
)
display(uncertainty)


**06. Visualise Matched Test Performance and Coverage**

Display participant-level observed errors and sensitivity. The plotted points are actual recorded Test outputs, not simulated gait signals.


In [ ]:
# ======================== 06. Visualise Test Evidence ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
fig, axes = plt.subplots(1, 2, figsize=(11.8,4.4))
for model, frame in person_results.groupby("model_id"):
    axes[0].scatter([model] * len(frame), frame.timing_mae_ms,
                    alpha=0.65, label=model)
    axes[1].scatter([model] * len(frame), frame.sensitivity,
                    alpha=0.65, label=model)
axes[0].set(
    title=("Held-out source-phase annotation agreement" if
           ANALYSIS_MODE == "source_phase_annotation"
           else "Held-out independent force-event timing"),
    ylabel="Matched event time difference (ms)",
)
axes[1].set(
    title=("Source-phase annotation coverage" if
           ANALYSIS_MODE == "source_phase_annotation"
           else "Force-event coverage"),
    ylabel="One-to-one event matching fraction",
)
for ax in axes:
    ax.grid(alpha=0.2)

# ------------------------ 2. Export Scientific Figure ------------------------
fig.tight_layout()
save_figure(fig, output_dir / f"{prefix}_test_participant_comparison.png")
plt.show()


**07. Final Scientific Interpretation and Boundaries**

The participant-independent Test supports only an offline within-dataset gait-event comparison for the available Camargo conditions. It cannot establish patient performance, real-time deployment or cross-device generalisation. The reported final Test result must come from the tables written above, not from planned values.


In [ ]:
# ======================== 07. Interpret Final Results ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
print("Selected Development method:", model_name)
print("Participant-disjoint reserved Test people:", len(test_people))
print("Target provenance:", TARGET)
if ANALYSIS_MODE == "source_phase_annotation":
    print("Interpret as agreement with processed source annotations only.")
    print("Only two independent Test people: no strong generalisation claim.")
    print("No force-calibrated reference IC validation was performed.")
print("Observed eligible Test trials:", len(trials))
print("Primary recorded Test file:",
      output_dir / f"{prefix}_test_participant_metrics.csv")
print("Interpret timing MAE beside sensitivity, precision and missed events.")
print("No raw-force signal was used to fit a Test-time IMU detector.")


**08. Research Summary and Output Download**

The saved `05_` results describe the frozen participant-independent Test, including event support, timing errors, participant summaries and descriptive uncertainty. Only results from actual execution are reportable; no Test-specific retuning occurs.

The shared helper packages the saved `05_` research tables and figures into one ZIP **outside the repository**. Retrieve it from Colab's file browser. Raw Camargo archives are not included, and no output is pushed to GitHub automatically.


In source-annotation mode, all Test outputs are written using the isolated `05_phase_` prefix, including the per-person results, paired uncertainty and the performance figure. The reserved Test set contains only two participants, so confidence claims must remain appropriately limited.


In [ ]:
# ================= 08. Research Summary and Output Download ==================

# ------------------- 1. Review Current Scientific Outputs --------------------
print('Frozen Test scientific evidence saved to:', output_dir)

# ------------------- 2. Package Notebook-Specific Evidence -------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
